# ASAP8 VIP somatic electrophysiology + longitudinal ROI registration

This notebook is a lab-meeting-oriented analysis pass for **ASAP8+ VIP somatic voltage recordings from one mouse across days**.

It uses the project **session registry as the source of truth** for session selection, asset paths, cortical depth, stimulus/image-set identity, and longitudinal ordering. Expression checks and other non-analysis session types are excluded before any files are loaded.

It is designed to answer four questions:

1. What kinds of spike/event phenotypes are present in ASAP8+ VIP interneurons?
2. Do neurons recorded at different cortical depths differ in fast spikes, compound events, and plateau-like depolarizations?
3. Which ROIs correspond to the same neuron across sessions?
4. Once registered, how stable are single-cell electrophysiological features across image-set days?

The detector is intentionally conservative and visual-first. Tune thresholds until overlay plots match what you would manually call spikes, compound events, and plateau-like depolarizations.


## 0. Imports and plotting setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os, re, json, glob, warnings
from datetime import datetime

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage, signal, optimize, stats
from IPython.display import display, HTML

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.extraction import load_voltage_roi_transform_h5
from vip_slap2_analysis.voltage import analysis
from vip_slap2_analysis.utils.utils import save_figure

display(HTML("<style>.container { width:100% !important; }</style>"))

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "font.size": 11,
})


## 1. Configure registry selection, nomenclature, and outputs

The registry remains the source of truth. The notebook:

- loads sessions with `VIPSessionRegistry`;
- excludes `expression_check` and `volume_imaging` sessions;
- resolves each selected row to a session asset;
- propagates registry/asset metadata into ROI-level outputs.

### Recommended registry columns for the new nomenclature

Add these columns to the registry `.xlsx`:

- `image_set`: short stable identity such as `A`, `B`, or `C`;
- `image_set_day_index`: zero-based exposure/session index within that image set (`0`, `1`, `2`, ...).

The notebook generates Unicode display labels such as **A₀**, **A₁**, and **B₀**. Keep `session_id` as the immutable join key; use `session_label` only for display and ordering.


In [ ]:
today_str = datetime.today().strftime("%Y-%m-%d")

BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")

# Keep this notebook scoped to one animal for the lab meeting.
TARGET_MICE = [852835]

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

# Optional exact session restriction after registry filtering.
SELECTED_SESSION_IDS = None

# Voltage extraction writes files like:
#   voltage_session_traces_dff_robust_f0_trial.h5
TRACE_VARIANT = "dff_robust_f0_trial"
SIGNAL = "dff"  # one of: "raw_f", "f0", "dff"

# Optional path override:
# - None: resolve separately from each asset
# - dict: {session_id: path}
# - format string containing {session_id}
SESSION_TRACE_H5 = None
SESSION_SUMMARY_H5 = None

# Registry nomenclature columns. Change these only if the workbook uses other names.
IMAGE_SET_COLUMN = "image_set"
IMAGE_SET_DAY_COLUMN = "image_set_day_index"
SESSION_LABEL_COLUMN = "session_label"

# Temporary fallback only, for running before the registry workbook is updated.
# Day indices are then assigned chronologically within each inferred image set.
# Set to {} to require the new registry columns.
LEGACY_SESSION_TYPE_TO_IMAGE_SET = {
    "familiar": "A",
    "novel": "B",
    "novel+": "B",
}

# None = first selected session. A session_id or generated label such as "A₀" is also accepted.
REFERENCE_SESSION = None

# ASAP8 is brightening: positive dF/F is depolarizing.
POLARITY = +1

# Used only when neither the timebase nor registry/summary metadata provides a rate.
FS_FALLBACK_HZ = 10_800.0

OUT_DIR = (
    BASE_PATH
    / "ASAP8"
    / str(TARGET_MICE[0])
    / "analysis"
    / "asap8_somatic_ephys_registration"
)
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
for p in [OUT_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SAVE_FIGURES = True
SAVE_TABLES = True

print("OUT_DIR:", OUT_DIR)


## 2. HDF5 / MATLAB summary readers

These readers support both extraction-style files (`dendriticVoltageSummary*.mat`, `dendriticVoltageTraces*.h5`) and newer full-session H5 outputs (`voltage_session_traces*.h5`) when present.

In [ ]:
def _as_scalar(x):
    arr = np.asarray(x)
    if arr.size == 0:
        return np.nan
    arr = np.squeeze(arr)
    if arr.size == 1:
        v = arr.reshape(-1)[0]
        return v.item() if isinstance(v, np.generic) else v
    return arr


def _decode_matlab_char(arr):
    arr = np.asarray(arr).squeeze()
    if arr.size == 0:
        return ""
    if arr.dtype.kind in "US":
        return "".join(arr.ravel().astype(str))
    return "".join(chr(int(c)) for c in arr.ravel() if int(c) != 0)


def _h5_exists(path, key):
    try:
        with h5py.File(path, "r") as f:
            return key in f
    except Exception:
        return False


def read_summary_scalar(summary_path, h5_path, default=np.nan):
    try:
        with h5py.File(summary_path, "r") as f:
            return _as_scalar(f[h5_path][()])
    except Exception:
        return default


def read_summary_text(summary_path, h5_path, default=""):
    try:
        with h5py.File(summary_path, "r") as f:
            return _decode_matlab_char(f[h5_path][()])
    except Exception:
        return default


def _read_matlab_ref(f, ref):
    if isinstance(ref, np.ndarray):
        ref = ref.reshape(-1)[0]
    return f[ref]


def orient_slap2_image_for_display(im, flip_y=True):
    im = np.asarray(im)
    if im.ndim == 3 and im.shape[-1] <= 4:
        out = im.transpose(1, 0, 2)
    else:
        out = im.T
    return np.flipud(out) if flip_y else out


def orient_slap2_masks_for_display(masks, flip_y=True):
    masks = np.asarray(masks)
    # summary/masks is commonly stored as n_roi x x x y
    if masks.ndim != 3:
        raise ValueError(f"Expected masks as 3D array, got {masks.shape}")
    out = masks.transpose(0, 2, 1)
    return out[:, ::-1, :] if flip_y else out


def read_ref_image(summary_path, dmd=1, for_display=True, flip_y=True):
    with h5py.File(summary_path, "r") as f:
        if "summary/refIM" in f:
            ref = f["summary/refIM"][int(dmd)-1, 0]
            im = np.asarray(_read_matlab_ref(f, ref)[()], dtype=np.float32)
        elif f"DMD{int(dmd)}/ref_image" in f:
            im = np.asarray(f[f"DMD{int(dmd)}/ref_image"][()], dtype=np.float32)
        else:
            raise KeyError("Could not find reference image in summary file")
    return orient_slap2_image_for_display(im, flip_y=flip_y) if for_display else im


def read_roi_masks(summary_path, dmd=1, for_display=True, flip_y=True):
    with h5py.File(summary_path, "r") as f:
        if "summary/masks" in f:
            ref = f["summary/masks"][int(dmd)-1, 0]
            masks = np.asarray(_read_matlab_ref(f, ref)[()], dtype=bool)
        elif f"DMD{int(dmd)}/roi_masks" in f:
            masks = np.asarray(f[f"DMD{int(dmd)}/roi_masks"][()], dtype=bool)
        else:
            raise KeyError("Could not find ROI masks in summary file")
    return orient_slap2_masks_for_display(masks, flip_y=flip_y) if for_display else masks


def get_dmd_metadata(summary_path, dmd=1):
    out = {"dmd": int(dmd)}
    candidate_paths = {
        "lineRateHz": [f"summary/metadata/DMD{dmd}/lineRateHz", f"summary/DMD{dmd}/lineRateHz", f"DMD{dmd}/lineRateHz"],
        "depth_um": [f"summary/metadata/DMD{dmd}/depth_um", f"DMD{dmd}/depth_um"],
        "n_rois": [f"summary/nROIs", f"DMD{dmd}/n_rois"],
    }
    with h5py.File(summary_path, "r") as f:
        for key, paths in candidate_paths.items():
            vals = []
            for p in paths:
                if p in f:
                    vals.append(_as_scalar(f[p][()]))
            out[key] = vals[0] if vals else np.nan
    if not np.isfinite(out.get("lineRateHz", np.nan)):
        out["lineRateHz"] = FS_FALLBACK_HZ
    return out

## 3. Load sessions through the registry

This follows the loading scheme from `ASAP8_ROI_Electrophysiological_Analysis.ipynb` rather than scanning mouse folders independently.

`process_df` is retained as a registry snapshot, while `sessions_df` adds resolved asset paths and standardized fields used throughout the notebook:

- `session_label`: e.g. A₀;
- `image_set` and `image_set_day_index`;
- `dmd1_depth_um` and `dmd2_depth_um`;
- `trace_h5_path` and `summary_path`;
- the original asset object for access to additional directories/metadata.


In [ ]:
SUBSCRIPT_TRANSLATION = str.maketrans(
    "0123456789-",
    "₀₁₂₃₄₅₆₇₈₉₋",
)


def unicode_subscript(value):
    """Convert an integer-like value to Unicode subscripts."""
    if pd.isna(value):
        return "?"
    try:
        value = int(float(value))
    except (TypeError, ValueError):
        return str(value)
    return str(value).translate(SUBSCRIPT_TRANSLATION)


def format_session_label(image_set, day_index):
    image_set = "" if pd.isna(image_set) else str(image_set).strip()
    return f"{image_set}{unicode_subscript(day_index)}" if image_set else f"session{unicode_subscript(day_index)}"


def _asset_attr(asset, name, default=None):
    value = getattr(asset, name, default)
    return value() if callable(value) else value


def _as_metadata_dict(asset):
    metadata = _asset_attr(asset, "metadata", {})
    return dict(metadata) if isinstance(metadata, dict) else {}


def _is_missing_scalar(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    try:
        missing = pd.isna(value)
        return bool(missing) if np.ndim(missing) == 0 else False
    except Exception:
        return False


def _first_nonmissing(mapping, keys, default=np.nan):
    for key in keys:
        if key in mapping and not _is_missing_scalar(mapping[key]):
            return mapping[key]
    return default


def _candidate_asset_roots(asset):
    roots = []
    for name in [
        "derived_dir", "qc_dir", "session_dir", "data_dir",
        "raw_dir", "extracted_dir", "root_dir",
    ]:
        value = _asset_attr(asset, name, None)
        if value is not None:
            roots.append(Path(value))
    # Preserve order while removing duplicates.
    return list(dict.fromkeys(roots))


def _resolve_override(override, session_id):
    if override is None:
        return None
    if isinstance(override, dict):
        value = override.get(str(session_id))
        return Path(value) if value else None
    value = str(override)
    if "{session_id}" in value:
        value = value.format(session_id=session_id)
    return Path(value)


def _find_asset_file(asset, exact_names=(), patterns=(), override=None):
    session_id = str(_asset_attr(asset, "session_id", ""))
    direct = _resolve_override(override, session_id)
    if direct is not None:
        return direct

    roots = _candidate_asset_roots(asset)
    search_roots = []
    for root in roots:
        search_roots.extend([root / "voltage", root])
    search_roots = list(dict.fromkeys(search_roots))

    for root in search_roots:
        for name in exact_names:
            candidate = root / name
            if candidate.exists():
                return candidate

    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            matches = sorted(root.glob(pattern))
            if matches:
                return matches[0]

    # Limited recursive fallback under derived directories.
    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            matches = sorted(root.glob(f"**/{pattern}"))
            if matches:
                return matches[0]
    return None


def resolve_trace_path(asset):
    exact = [f"voltage_session_traces_{TRACE_VARIANT}.h5"]
    patterns = [
        f"voltage_session_traces_{TRACE_VARIANT}.h5",
        "voltage_session_traces*.h5",
        "dendriticVoltageTraces*.h5",
    ]
    return _find_asset_file(
        asset,
        exact_names=exact,
        patterns=patterns,
        override=SESSION_TRACE_H5,
    )


def resolve_summary_path(asset):
    patterns = [
        "voltage_roi_transform*.h5",
        "*roi*transform*.h5",
        "dendriticVoltageSummary*.mat",
        "dendriticVoltageSummary*.h5",
        "*voltage*summary*.h5",
    ]
    return _find_asset_file(
        asset,
        patterns=patterns,
        override=SESSION_SUMMARY_H5,
    )


def infer_session_datetime(session_id):
    match = re.search(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})", str(session_id))
    if not match:
        return pd.NaT
    return pd.to_datetime(match.group(1), format="%Y-%m-%d_%H-%M-%S", errors="coerce")


registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=TARGET_MICE,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
).copy()

if SELECTED_SESSION_IDS is not None:
    wanted = {str(x) for x in SELECTED_SESSION_IDS}
    session_col = "session_id" if "session_id" in process_df.columns else "id"
    process_df = process_df[process_df[session_col].astype(str).isin(wanted)].copy()

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

session_rows = []
for registry_order, ((registry_index, registry_row), asset) in enumerate(zip(process_df.iterrows(), assets)):
    registry_values = registry_row.to_dict()
    asset_metadata = _as_metadata_dict(asset)

    # Registry columns win; asset.metadata fills fields absent from the table.
    combined = dict(asset_metadata)
    combined.update({
        key: value
        for key, value in registry_values.items()
        if not _is_missing_scalar(value)
    })

    session_id = str(_asset_attr(asset, "session_id", _first_nonmissing(combined, ["session_id", "id"], "")))
    subject_id = _asset_attr(asset, "subject_id", _first_nonmissing(combined, ["subject_id", "mouse_id"], np.nan))
    session_type = _first_nonmissing(combined, ["session_type", "type"], "")
    paradigm = _first_nonmissing(combined, ["paradigm", "behavior_paradigm"], "")

    image_set = _first_nonmissing(
        combined,
        [IMAGE_SET_COLUMN, "stimulus_image_set", "image_set_name"],
        np.nan,
    )
    image_set_day = _first_nonmissing(
        combined,
        [IMAGE_SET_DAY_COLUMN, "image_day_index", "stimulus_set_day_index"],
        np.nan,
    )
    explicit_label = _first_nonmissing(
        combined,
        [SESSION_LABEL_COLUMN],
        np.nan,
    )

    if pd.isna(image_set) or str(image_set).strip() == "":
        image_set = LEGACY_SESSION_TYPE_TO_IMAGE_SET.get(str(session_type).strip().lower(), np.nan)

    session_rows.append({
        **registry_values,
        "registry_index": registry_index,
        "registry_order": registry_order,
        "session_id": session_id,
        "subject_id": subject_id,
        "session_type": session_type,
        "paradigm": paradigm,
        "session_datetime": infer_session_datetime(session_id),
        "image_set": image_set,
        "image_set_day_index": pd.to_numeric(image_set_day, errors="coerce"),
        "explicit_session_label": explicit_label,
        "dmd1_depth_um": pd.to_numeric(_first_nonmissing(
            combined,
            ["dmd1_depth_um", "dmd1_depth", "DMD1_depth", "depth_dmd1"],
            np.nan,
        ), errors="coerce"),
        "dmd2_depth_um": pd.to_numeric(_first_nonmissing(
            combined,
            ["dmd2_depth_um", "dmd2_depth", "DMD2_depth", "depth_dmd2"],
            np.nan,
        ), errors="coerce"),
        "asset": asset,
        "derived_dir": _asset_attr(asset, "derived_dir", None),
        "qc_dir": _asset_attr(asset, "qc_dir", None),
        "trace_h5_path": resolve_trace_path(asset),
        "summary_path": resolve_summary_path(asset),
        "asset_metadata": asset_metadata,
    })

sessions_df = pd.DataFrame(session_rows)

if sessions_df.empty:
    raise RuntimeError("The registry query returned no analysis sessions.")

# Registry output is generally already chronological, but session datetime is a safer
# longitudinal ordering when present.
sessions_df = (
    sessions_df
    .sort_values(["subject_id", "session_datetime", "registry_order"], na_position="last")
    .reset_index(drop=True)
)
sessions_df["session_order"] = sessions_df.groupby("subject_id").cumcount()

# Before the workbook is updated, derive missing zero-based day indices within image set.
derived_day_index = sessions_df.groupby(
    ["subject_id", "image_set"],
    dropna=False,
).cumcount()
sessions_df["image_set_day_index"] = (
    sessions_df["image_set_day_index"]
    .fillna(derived_day_index)
    .astype("Int64")
)

generated_label = [
    format_session_label(image_set, day_index)
    for image_set, day_index in zip(
        sessions_df["image_set"],
        sessions_df["image_set_day_index"],
    )
]
explicit = sessions_df["explicit_session_label"].fillna("").astype(str).str.strip()
sessions_df["session_label"] = np.where(explicit.ne(""), explicit, generated_label)

# Resolve a reference by immutable session_id or by display label.
if REFERENCE_SESSION is None:
    REFERENCE_SESSION_ID = sessions_df.loc[0, "session_id"]
else:
    ref = str(REFERENCE_SESSION)
    by_id = sessions_df["session_id"].astype(str).eq(ref)
    by_label = sessions_df["session_label"].astype(str).eq(ref)
    matches = sessions_df[by_id | by_label]
    if len(matches) != 1:
        raise ValueError(
            f"REFERENCE_SESSION={ref!r} matched {len(matches)} sessions; "
            "provide one unique session_id or session_label."
        )
    REFERENCE_SESSION_ID = matches.iloc[0]["session_id"]

# Columns propagated into ROI-level and registration outputs.
SESSION_CONTEXT_COLUMNS = [
    "subject_id",
    "session_id",
    "session_label",
    "session_order",
    "session_datetime",
    "session_type",
    "paradigm",
    "image_set",
    "image_set_day_index",
    "dmd1_depth_um",
    "dmd2_depth_um",
]


def session_context(row, dmd=None):
    out = {key: row.get(key, np.nan) for key in SESSION_CONTEXT_COLUMNS}
    if dmd is not None:
        out["depth_um"] = row.get(f"dmd{int(dmd)}_depth_um", np.nan)
    return out


def session_label_for_id(session_id):
    match = sessions_df.loc[
        sessions_df["session_id"].astype(str).eq(str(session_id)),
        "session_label",
    ]
    return match.iloc[0] if len(match) else str(session_id)


def registry_sampling_rate(row, dmd=None):
    metadata = row.get("asset_metadata", {})
    candidates = []
    if dmd is not None:
        candidates.extend([
            f"dmd{dmd}_sample_rate_hz",
            f"dmd{dmd}_sampling_rate_hz",
            f"dmd{dmd}_line_rate_hz",
            f"dmd{dmd}_lineRateHz",
        ])
    candidates.extend([
        "sample_rate_hz",
        "sampling_rate_hz",
        "line_rate_hz",
        "lineRateHz",
        "fs_hz",
    ])
    combined = dict(metadata) if isinstance(metadata, dict) else {}
    combined.update(row.to_dict())
    value = pd.to_numeric(_first_nonmissing(combined, candidates, np.nan), errors="coerce")
    return float(value) if np.isfinite(value) else np.nan


display_columns = [
    "subject_id", "session_label", "session_id", "session_type", "paradigm",
    "image_set", "image_set_day_index", "dmd1_depth_um", "dmd2_depth_um",
    "trace_h5_path", "summary_path",
]
display(sessions_df[display_columns])
print(f"Found {len(sessions_df)} analysis sessions after registry filtering.")
print("Reference session:", session_label_for_id(REFERENCE_SESSION_ID), REFERENCE_SESSION_ID)

missing_trace = sessions_df["trace_h5_path"].isna()
missing_summary = sessions_df["summary_path"].isna()
if missing_trace.any():
    warnings.warn(
        "No trace H5 was resolved for: "
        + ", ".join(sessions_df.loc[missing_trace, "session_id"].astype(str))
    )
if missing_summary.any():
    warnings.warn(
        "No ROI transform/summary file was resolved for: "
        + ", ".join(sessions_df.loc[missing_summary, "session_id"].astype(str))
    )

duplicate_labels = sessions_df.duplicated(["subject_id", "session_label"], keep=False)
if duplicate_labels.any():
    display(sessions_df.loc[duplicate_labels, ["subject_id", "session_id", "session_label"]])
    warnings.warn("Duplicate session labels detected. Update image_set/day indices in the registry.")

if SAVE_TABLES:
    export_cols = [c for c in sessions_df.columns if c not in {"asset", "asset_metadata"}]
    sessions_df[export_cols].to_csv(TABLE_DIR / "selected_session_registry_snapshot.csv", index=False)


## 4. Trace loader

Output is a dictionary: `{dmd: roi x samples}` plus a common timebase. The code prefers already-computed `dff`, then falls back to raw extraction traces. For ASAP8, depolarizing events should be positive after applying `POLARITY = +1`.

In [ ]:
def _dataset_to_roi_by_time(arr):
    arr = np.asarray(arr)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D trace array, got {arr.shape}")
    # Heuristic: ROI dimension is smaller than time dimension.
    if arr.shape[0] <= arr.shape[1]:
        return arr.astype(np.float32)
    return arr.T.astype(np.float32)


def describe_h5_tree(path, max_items=60):
    rows = []
    with h5py.File(path, "r") as f:
        def visitor(name, obj):
            if len(rows) >= max_items:
                return
            if isinstance(obj, h5py.Dataset):
                rows.append({"name": name, "shape": obj.shape, "dtype": str(obj.dtype)})
        f.visititems(visitor)
    return pd.DataFrame(rows)


def load_voltage_traces(trace_h5_path, fs=FS_FALLBACK_HZ, polarity=+1):
    trace_h5_path = Path(trace_h5_path)
    traces = {}
    info = {"mode": None, "fs": fs, "time_sec": None}
    with h5py.File(trace_h5_path, "r") as f:
        # Newer full-session H5: /DMD1/dff, /DMD1/timebase_sec
        dmd_keys = [k for k in f.keys() if re.fullmatch(r"DMD\d+", k)]
        if dmd_keys:
            for k in sorted(dmd_keys):
                dmd = int(k.replace("DMD", ""))
                signal_key = None
                for candidate in ["dff", "raw_f", "raw", "trace"]:
                    if f"{k}/{candidate}" in f:
                        signal_key = f"{k}/{candidate}"
                        break
                if signal_key is None:
                    continue
                traces[dmd] = polarity * _dataset_to_roi_by_time(f[signal_key][()])
                if f"{k}/timebase_sec" in f and info["time_sec"] is None:
                    info["time_sec"] = np.asarray(f[f"{k}/timebase_sec"][()], dtype=float).reshape(-1)
            if traces:
                info["mode"] = "full_session_h5"

        # Extraction continuous: /traces/continuous/DMD#
        if not traces and "traces/continuous" in f:
            for key in sorted(f["traces/continuous"].keys()):
                if not key.startswith("DMD"):
                    continue
                dmd = int(key.replace("DMD", ""))
                traces[dmd] = polarity * _dataset_to_roi_by_time(f[f"traces/continuous/{key}"][()])
            if traces:
                info["mode"] = "extraction_continuous"

        # Extraction trial-only: /traces/trial_XXXX/DMD#
        if not traces and "traces" in f:
            trial_keys = sorted([k for k in f["traces"].keys() if k.startswith("trial_")])
            if trial_keys:
                dmd_names = sorted([k for k in f[f"traces/{trial_keys[0]}"] if k.startswith("DMD")])
                for dmd_name in dmd_names:
                    dmd = int(dmd_name.replace("DMD", ""))
                    parts = [_dataset_to_roi_by_time(f[f"traces/{tk}/{dmd_name}"][()]) for tk in trial_keys]
                    traces[dmd] = polarity * np.concatenate(parts, axis=1)
                info["mode"] = "trial_concatenated"

    if not traces:
        raise KeyError(f"Could not find usable voltage traces in {trace_h5_path}")
    n_samples = min(X.shape[1] for X in traces.values())
    traces = {dmd: X[:, :n_samples] for dmd, X in traces.items()}
    if info["time_sec"] is None or len(info["time_sec"]) != n_samples:
        info["time_sec"] = np.arange(n_samples) / float(fs)
    else:
        # Use median delta to update fs if available.
        dt = np.nanmedian(np.diff(info["time_sec"]))
        if np.isfinite(dt) and dt > 0:
            info["fs"] = 1.0 / dt
    return traces, info

## 5. Spike/event feature extraction

Core phenotype columns:

- **spike_rate_hz**: detected fast-event rate.
- **median_width_ms**: median full-width at half maximum of fast events.
- **median_rise10_90_ms** and **median_decay50_ms**: waveform kinetics.
- **median_plateau_index**: sustained post-peak depolarization relative to peak height. Larger means more compound/plateau-like.
- **compound_event_fraction**: fraction of peaks with another peak nearby.
- **plateau_burden_fraction**: fraction of time spent in slow/envelope-defined depolarized epochs.
- **isi_cv**, **burst_fraction_20ms**, **burst_fraction_50ms**: spike-train irregularity and bursting.
- **noise_mad**, **snr_median_peak**, **spectral_centroid_hz**, **bandpower_*:** QC/biophysical context.

In [ ]:
def rolling_percentile_baseline(x, fs, window_s=2.0, percentile=20):
    size = int(round(window_s * fs))
    size = max(size, 101)
    if size % 2 == 0:
        size += 1
    # percentile_filter is robust but can be slow; for large arrays this is still ok for tens of ROIs.
    return ndimage.percentile_filter(x, percentile=percentile, size=size, mode="nearest")


def robust_mad(x):
    x = np.asarray(x, float)
    med = np.nanmedian(x)
    return 1.4826 * np.nanmedian(np.abs(x - med))


def preprocess_for_spikes(x, fs, baseline_window_s=2.0, hp_hz=20.0, lp_hz=1200.0):
    x = np.asarray(x, float)
    baseline = rolling_percentile_baseline(x, fs, window_s=baseline_window_s, percentile=20)
    xd = x - baseline
    nyq = fs / 2
    lo = max(0.001, hp_hz / nyq)
    hi = min(0.99, lp_hz / nyq)
    if lo < hi:
        sos = signal.butter(3, [lo, hi], btype="bandpass", output="sos")
        x_spk = signal.sosfiltfilt(sos, xd)
    else:
        x_spk = xd
    return xd, x_spk, baseline


def detect_fast_events(x_spk, fs, z_thresh=5.0, min_distance_ms=3.0, prominence_z=3.0):
    noise = robust_mad(x_spk)
    if not np.isfinite(noise) or noise <= 0:
        return np.array([], dtype=int), {"noise_mad": noise, "threshold": np.nan}
    distance = max(1, int(round(min_distance_ms / 1000 * fs)))
    peaks, props = signal.find_peaks(
        x_spk,
        height=z_thresh * noise,
        prominence=prominence_z * noise,
        distance=distance,
    )
    return peaks.astype(int), {"noise_mad": noise, "threshold": z_thresh * noise, **props}


def _local_crossing(y, center, target, direction=-1, max_search=500):
    # direction=-1 searches left; +1 searches right
    n = len(y)
    i = int(center)
    steps = range(i, max(-1, i - max_search), -1) if direction < 0 else range(i, min(n, i + max_search))
    prev = None
    for j in steps:
        val = y[j]
        if prev is not None:
            if (prev - target) * (val - target) <= 0:
                return float(j)
        prev = val
    return np.nan


def event_waveform_metrics(xd, x_spk, peaks, fs, window_ms=(-20, 120)):
    rows = []
    pre = int(round(abs(window_ms[0]) / 1000 * fs))
    post = int(round(window_ms[1] / 1000 * fs))
    t_ms = (np.arange(-pre, post + 1) / fs) * 1000
    snippets = []
    for p in np.asarray(peaks, dtype=int):
        if p - pre < 0 or p + post + 1 > len(xd):
            continue
        s = xd[p - pre:p + post + 1].astype(float)
        sf = x_spk[p - pre:p + post + 1].astype(float)
        base = np.nanmedian(s[:max(3, int(0.25 * pre))]) if pre > 5 else np.nanmedian(s[:pre])
        s0 = s - base
        peak = s0[pre]
        peak_fast = sf[pre]
        if not np.isfinite(peak) or peak <= 0:
            peak = np.nanmax(s0[max(0, pre-3):min(len(s0), pre+4)])
        half = 0.5 * peak if np.isfinite(peak) else np.nan
        width_ms = np.nan
        if np.isfinite(half) and half > 0:
            left = _local_crossing(s0, pre, half, direction=-1, max_search=pre)
            right = _local_crossing(s0, pre, half, direction=+1, max_search=post)
            if np.isfinite(left) and np.isfinite(right):
                width_ms = (right - left) / fs * 1000
        # rise 10->90, approximate from left side
        rise_ms = np.nan
        if np.isfinite(peak) and peak > 0:
            left10 = _local_crossing(s0, pre, 0.1 * peak, direction=-1, max_search=pre)
            left90 = _local_crossing(s0, pre, 0.9 * peak, direction=-1, max_search=pre)
            if np.isfinite(left10) and np.isfinite(left90):
                rise_ms = abs(left90 - left10) / fs * 1000
        decay50_ms = np.nan
        if np.isfinite(peak) and peak > 0:
            right50 = _local_crossing(s0, pre, 0.5 * peak, direction=+1, max_search=post)
            if np.isfinite(right50):
                decay50_ms = (right50 - pre) / fs * 1000
        # Plateau index: 20-80 ms post-peak mean divided by peak amplitude.
        idx20 = pre + int(round(20 / 1000 * fs))
        idx80 = pre + int(round(80 / 1000 * fs))
        post_mean = np.nanmean(s0[idx20:min(idx80, len(s0))]) if idx20 < len(s0) else np.nan
        plateau_index = post_mean / peak if np.isfinite(peak) and peak > 0 else np.nan
        ahp_min = np.nanmin(s0[pre:min(len(s0), pre + int(round(80 / 1000 * fs)))]) if pre < len(s0) else np.nan
        rows.append({
            "sample": int(p),
            "peak_amp": peak,
            "peak_fast_amp": peak_fast,
            "width_ms": width_ms,
            "rise10_90_ms": rise_ms,
            "decay50_ms": decay50_ms,
            "plateau_index": plateau_index,
            "post20_80_mean": post_mean,
            "ahp_min": ahp_min,
        })
        snippets.append(s0)
    return pd.DataFrame(rows), t_ms, np.asarray(snippets)


def plateau_epochs(xd, fs, lp_hz=20.0, z_thresh=2.5, min_duration_ms=20.0):
    nyq = fs / 2
    if lp_hz / nyq < 0.99:
        sos = signal.butter(3, lp_hz / nyq, btype="lowpass", output="sos")
        xlow = signal.sosfiltfilt(sos, xd)
    else:
        xlow = xd
    noise = robust_mad(xlow)
    thr = z_thresh * noise
    above = xlow > thr
    lab, n = ndimage.label(above)
    min_len = int(round(min_duration_ms / 1000 * fs))
    rows = []
    for k in range(1, n + 1):
        idx = np.flatnonzero(lab == k)
        if len(idx) < min_len:
            above[idx] = False
            continue
        rows.append({
            "start_sample": int(idx[0]),
            "end_sample": int(idx[-1]),
            "duration_ms": len(idx) / fs * 1000,
            "max_amp": float(np.nanmax(xlow[idx])),
        })
    return pd.DataFrame(rows), xlow, above


def compute_spectral_features(xd, fs, bands=((0.2, 5), (5, 30), (30, 200), (200, 1000))):
    nperseg = int(min(max(round(4 * fs), 512), len(xd)))
    freqs, psd = signal.welch(xd, fs=fs, nperseg=nperseg, noverlap=nperseg//2)
    psd_sum = np.trapz(psd, freqs) if len(freqs) else np.nan
    centroid = np.trapz(freqs * psd, freqs) / psd_sum if np.isfinite(psd_sum) and psd_sum > 0 else np.nan
    out = {"spectral_centroid_hz": centroid, "total_power": psd_sum}
    for lo, hi in bands:
        sel = (freqs >= lo) & (freqs < hi)
        p = np.trapz(psd[sel], freqs[sel]) if np.any(sel) else np.nan
        out[f"bandpower_{lo:g}_{hi:g}_Hz"] = p
        out[f"fracpower_{lo:g}_{hi:g}_Hz"] = p / psd_sum if np.isfinite(psd_sum) and psd_sum > 0 else np.nan
    return out


def summarize_roi_events(x, fs, z_thresh=5.0, prominence_z=3.0):
    xd, x_spk, baseline = preprocess_for_spikes(x, fs)
    peaks, det = detect_fast_events(x_spk, fs, z_thresh=z_thresh, prominence_z=prominence_z)
    event_df, sta_t_ms, snippets = event_waveform_metrics(xd, x_spk, peaks, fs)
    pl_df, xlow, plateau_mask = plateau_epochs(xd, fs)
    duration_s = len(x) / fs
    isi_ms = np.diff(peaks) / fs * 1000 if len(peaks) > 1 else np.array([])
    compound_20 = np.r_[False, isi_ms < 20] | np.r_[isi_ms < 20, False] if len(peaks) > 1 else np.zeros(len(peaks), bool)
    compound_50 = np.r_[False, isi_ms < 50] | np.r_[isi_ms < 50, False] if len(peaks) > 1 else np.zeros(len(peaks), bool)
    summary = {
        "n_samples": len(x),
        "duration_s": duration_s,
        "n_spikes": len(peaks),
        "spike_rate_hz": len(peaks) / duration_s if duration_s > 0 else np.nan,
        "noise_mad": det.get("noise_mad", np.nan),
        "detect_threshold": det.get("threshold", np.nan),
        "median_peak_amp": event_df["peak_amp"].median() if not event_df.empty else np.nan,
        "p90_peak_amp": event_df["peak_amp"].quantile(0.9) if not event_df.empty else np.nan,
        "median_width_ms": event_df["width_ms"].median() if not event_df.empty else np.nan,
        "median_rise10_90_ms": event_df["rise10_90_ms"].median() if not event_df.empty else np.nan,
        "median_decay50_ms": event_df["decay50_ms"].median() if not event_df.empty else np.nan,
        "median_plateau_index": event_df["plateau_index"].median() if not event_df.empty else np.nan,
        "p90_plateau_index": event_df["plateau_index"].quantile(0.9) if not event_df.empty else np.nan,
        "compound_event_fraction_20ms": float(np.mean(compound_20)) if len(compound_20) else np.nan,
        "compound_event_fraction_50ms": float(np.mean(compound_50)) if len(compound_50) else np.nan,
        "isi_median_ms": np.nanmedian(isi_ms) if isi_ms.size else np.nan,
        "isi_cv": np.nanstd(isi_ms) / np.nanmean(isi_ms) if isi_ms.size and np.nanmean(isi_ms) > 0 else np.nan,
        "plateau_event_rate_hz": len(pl_df) / duration_s if duration_s > 0 else np.nan,
        "plateau_burden_fraction": float(np.mean(plateau_mask)) if len(plateau_mask) else np.nan,
        "median_plateau_duration_ms": pl_df["duration_ms"].median() if not pl_df.empty else np.nan,
    }
    summary["snr_median_peak"] = summary["median_peak_amp"] / summary["noise_mad"] if summary["noise_mad"] and np.isfinite(summary["noise_mad"]) else np.nan
    summary.update(compute_spectral_features(xd, fs))
    aux = {
        "xd": xd, "x_spk": x_spk, "baseline": baseline, "peaks": peaks,
        "event_df": event_df, "sta_t_ms": sta_t_ms, "snippets": snippets,
        "xlow": xlow, "plateau_df": pl_df, "plateau_mask": plateau_mask,
    }
    return summary, aux

## 6. Run electrophysiology metrics across registry-selected sessions/ROIs

The loop below uses registry-resolved paths and propagates session context into every ROI row. Sampling rate is resolved in this order:

1. explicit timebase in the trace H5;
2. registry/asset metadata;
3. ROI summary metadata;
4. `FS_FALLBACK_HZ`.

Start with conservative event thresholds, then validate them using the overlay plots.


In [ ]:
Z_THRESH = 5.0
PROMINENCE_Z = 3.0
MAX_SESSIONS_FOR_AUX_CACHE = 3  # retain snippets for only a few sessions

metric_rows = []
aux_cache = {}
failures = []

for i, row in sessions_df.iterrows():
    session_id = str(row["session_id"])
    session_label = row["session_label"]
    print(f"[{i+1}/{len(sessions_df)}] {session_label}: {session_id}")

    trace_path = row["trace_h5_path"]
    summary_path = row["summary_path"]

    if trace_path is None or pd.isna(trace_path):
        failures.append({
            **session_context(row),
            "error": "No trace_h5_path resolved from session asset.",
        })
        continue

    try:
        fs = registry_sampling_rate(row, dmd=1)

        if not np.isfinite(fs) and summary_path is not None and not pd.isna(summary_path):
            try:
                md1 = get_dmd_metadata(summary_path, dmd=1)
                fs = float(md1.get("lineRateHz", np.nan))
            except Exception:
                fs = np.nan

        if not np.isfinite(fs):
            fs = FS_FALLBACK_HZ

        traces, info = load_voltage_traces(trace_path, fs=fs, polarity=POLARITY)
        fs = float(info["fs"])

        if i < MAX_SESSIONS_FOR_AUX_CACHE:
            aux_cache[session_id] = {
                "traces": traces,
                "info": info,
                "aux": {},
                "session_label": session_label,
            }

        for dmd, X in sorted(traces.items()):
            masks = None
            if summary_path is not None and not pd.isna(summary_path):
                try:
                    masks = read_roi_masks(summary_path, dmd=dmd, for_display=True)
                except Exception as exc:
                    warnings.warn(f"Could not read masks for {session_label} DMD{dmd}: {exc}")

            for roi_idx, x in enumerate(X):
                summary, aux = summarize_roi_events(
                    x,
                    fs,
                    z_thresh=Z_THRESH,
                    prominence_z=PROMINENCE_Z,
                )
                roi_area = (
                    float(np.sum(masks[roi_idx]))
                    if masks is not None and roi_idx < len(masks)
                    else np.nan
                )

                metric_rows.append({
                    **session_context(row, dmd=dmd),
                    "dmd": int(dmd),
                    "roi": int(roi_idx),
                    "roi_label": f"{session_label}_DMD{dmd}_ROI{roi_idx}",
                    "trace_h5_path": str(trace_path),
                    "trace_mode": info["mode"],
                    "fs_hz": fs,
                    "roi_area_px": roi_area,
                    **summary,
                })

                if session_id in aux_cache:
                    aux_cache[session_id]["aux"][(int(dmd), int(roi_idx))] = aux

    except Exception as exc:
        failures.append({
            **session_context(row),
            "error": repr(exc),
        })
        warnings.warn(f"Failed {session_label} ({session_id}): {exc}")

metrics_df = pd.DataFrame(metric_rows)
failures_df = pd.DataFrame(failures)

display(metrics_df.head())
print(metrics_df.shape)
if not failures_df.empty:
    display(failures_df)

if SAVE_TABLES:
    metrics_df.to_csv(TABLE_DIR / "asap8_roi_ephys_metrics.csv", index=False)
    if not failures_df.empty:
        failures_df.to_csv(TABLE_DIR / "asap8_metric_failures.csv", index=False)


## 7. Metric overview plots for the first talk section

In [ ]:
PLOT_METRICS = [
    "spike_rate_hz",
    "median_width_ms",
    "median_plateau_index",
    "compound_event_fraction_50ms",
    "plateau_burden_fraction",
    "snr_median_peak",
]

if metrics_df.empty:
    raise RuntimeError("Run the metric extraction cell first.")

# Per-session/DMD distributions. Session labels come directly from image set/day metadata.
for metric in PLOT_METRICS:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    xlabels, positions, data = [], [], []
    pos = 0

    ordered_sessions = (
        metrics_df[["session_id", "session_label", "session_order"]]
        .drop_duplicates()
        .sort_values("session_order")
    )

    for _, session_info in ordered_sessions.iterrows():
        sid = session_info["session_id"]
        sub_sess = metrics_df[metrics_df["session_id"].eq(sid)]
        for dmd, sub in sub_sess.groupby("dmd", sort=True):
            vals = sub[metric].dropna().values
            if not len(vals):
                continue
            depth = sub["depth_um"].dropna()
            depth_text = f"\n{depth.iloc[0]:.0f} µm" if len(depth) else ""
            data.append(vals)
            positions.append(pos)
            xlabels.append(f"{session_info['session_label']}\nDMD{dmd}{depth_text}")
            pos += 1
        pos += 0.5

    ax.boxplot(data, positions=positions, widths=0.6, showfliers=False)
    rng = np.random.default_rng(0)
    for p, vals in zip(positions, data):
        jitter = rng.normal(0, 0.05, size=len(vals))
        ax.plot(np.full(len(vals), p) + jitter, vals, "o", ms=4, alpha=0.7)

    ax.set_xticks(positions)
    ax.set_xticklabels(xlabels, rotation=45, ha="right")
    ax.set_ylabel(metric)
    ax.set_title(f"ROI-level {metric} by image-set day and recording depth")
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(FIG_DIR / f"metric_distribution_{metric}.png", bbox_inches="tight")
    plt.show()

# Compact phenotype scatter.
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for dmd, sub in metrics_df.groupby("dmd"):
    depth = sub["depth_um"].median()
    depth_text = f", median depth {depth:.0f} µm" if np.isfinite(depth) else ""
    ax.scatter(
        sub["spike_rate_hz"],
        sub["median_plateau_index"],
        s=60,
        alpha=0.75,
        label=f"DMD{dmd}{depth_text}",
    )
ax.set_xlabel("Spike/event rate (Hz)")
ax.set_ylabel("Median plateau index")
ax.set_title("Fast-event rate vs compound/plateau phenotype")
ax.legend(frameon=False)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIG_DIR / "phenotype_rate_vs_plateau.png", bbox_inches="tight")
plt.show()


## 8. Detection QC overlay plots

This is the most important validation plot. For a few ROIs, confirm that red markers correspond to believable ASAP8 spikes/compound events. If not, tune `Z_THRESH`, `PROMINENCE_Z`, and preprocessing settings above.

In [ ]:
def plot_detection_overlay(session_id, dmd=1, roi=0, tlim_s=(0, 10)):
    if session_id not in aux_cache:
        raise KeyError(f"{session_id} not cached. Increase MAX_SESSIONS_FOR_AUX_CACHE or rerun only this session.")
    cache = aux_cache[session_id]
    fs = cache["info"]["fs"]
    x = cache["traces"][int(dmd)][int(roi)]
    aux = cache["aux"][(int(dmd), int(roi))]
    t = np.arange(len(x)) / fs
    sel = (t >= tlim_s[0]) & (t <= tlim_s[1])
    peaks = aux["peaks"]
    psel = peaks[(peaks / fs >= tlim_s[0]) & (peaks / fs <= tlim_s[1])]

    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axes[0].plot(t[sel], x[sel], lw=0.7)
    axes[0].plot(psel / fs, x[psel], "ro", ms=4, label="detected")
    axes[0].set_ylabel("dF/F")
    axes[0].legend(frameon=False)
    axes[0].set_title(f"{session_label_for_id(session_id)} DMD{dmd} ROI{roi}: raw/depolarizing trace")

    axes[1].plot(t[sel], aux["x_spk"][sel], lw=0.7)
    axes[1].axhline(aux["event_df"]["peak_fast_amp"].median() if not aux["event_df"].empty else 0, ls=":", lw=0.8)
    axes[1].plot(psel / fs, aux["x_spk"][psel], "ro", ms=4)
    axes[1].set_ylabel("spike-filtered")
    axes[1].set_xlabel("time (s)")
    fig.tight_layout()
    return fig, axes

# Example: first cached session, first DMD/ROI.
example_session = list(aux_cache.keys())[0]
fig, axes = plot_detection_overlay(example_session, dmd=1, roi=0, tlim_s=(0, 10))
if SAVE_FIGURES:
    fig.savefig(FIG_DIR / "example_detection_overlay.png", bbox_inches="tight")
plt.show()

## 9. Event waveform examples and spike-triggered averages

In [ ]:
def plot_sta_grid(session_id, max_rois=12):
    cache = aux_cache[session_id]
    keys = sorted(cache["aux"].keys())[:max_rois]
    n = len(keys)
    fig, axes = plt.subplots(n, 1, figsize=(8, max(2.0*n, 3)), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, (dmd, roi) in zip(axes, keys):
        aux = cache["aux"][(dmd, roi)]
        snip = aux["snippets"]
        t_ms = aux["sta_t_ms"]
        if snip.size:
            if len(snip) > 200:
                rng = np.random.default_rng(0)
                snip_plot = snip[rng.choice(len(snip), size=200, replace=False)]
            else:
                snip_plot = snip
            ax.plot(t_ms, snip_plot.T, color="0.75", lw=0.4, alpha=0.35)
            ax.plot(t_ms, np.nanmedian(snip, axis=0), color="k", lw=1.5)
        ax.axvline(0, color="r", ls="--", lw=0.8)
        ax.set_ylabel(f"DMD{dmd}\nROI{roi}")
    axes[-1].set_xlabel("time from detected event (ms)")
    fig.suptitle(f"Spike/event-triggered waveforms — {session_label_for_id(session_id)}", y=1.01)
    fig.tight_layout()
    return fig, axes

fig, axes = plot_sta_grid(example_session, max_rois=12)
if SAVE_FIGURES:
    fig.savefig(FIG_DIR / "example_sta_grid.png", bbox_inches="tight")
plt.show()

## 10. Reference images and ROI masks for longitudinal registration

Images and masks are loaded from the ROI transform/summary file resolved for each registry asset. The montage uses image-set/day labels, while all exported tables retain `session_id`.


In [ ]:
def mask_centroids(masks):
    rows = []
    for i, m in enumerate(masks):
        yy, xx = np.nonzero(m)
        if len(xx) == 0:
            rows.append({"roi": i, "x": np.nan, "y": np.nan, "area_px": 0})
        else:
            rows.append({
                "roi": i,
                "x": float(np.mean(xx)),
                "y": float(np.mean(yy)),
                "area_px": int(len(xx)),
            })
    return pd.DataFrame(rows)


def overlay_roi_masks(ax, masks, label_prefix="", contour_lw=0.8):
    for roi_idx, mask in enumerate(masks):
        if np.any(mask):
            ax.contour(mask.astype(float), levels=[0.5], linewidths=contour_lw)
            yy, xx = np.nonzero(mask)
            ax.text(
                float(np.mean(xx)),
                float(np.mean(yy)),
                f"{label_prefix}{roi_idx}",
                ha="center",
                va="center",
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.65),
            )


roi_image_cache = {}
roi_geometry_rows = []

for _, row in sessions_df.iterrows():
    session_id = str(row["session_id"])
    session_label = row["session_label"]
    summary_path = row["summary_path"]
    roi_image_cache[session_id] = {}

    if summary_path is None or pd.isna(summary_path):
        continue

    for dmd in [1, 2]:
        try:
            ref = read_ref_image(summary_path, dmd=dmd, for_display=True)
            ref2 = ref[..., 0] if ref.ndim == 3 else ref
            masks = read_roi_masks(summary_path, dmd=dmd, for_display=True)
            roi_image_cache[session_id][dmd] = {
                "ref": ref2,
                "masks": masks,
                "session_label": session_label,
            }

            cen = mask_centroids(masks)
            for _, roi_row in cen.iterrows():
                roi_geometry_rows.append({
                    **session_context(row, dmd=dmd),
                    "dmd": dmd,
                    **roi_row.to_dict(),
                })

        except Exception as exc:
            warnings.warn(f"Could not load DMD{dmd} masks for {session_label}: {exc}")

roi_geometry_df = pd.DataFrame(roi_geometry_rows)
display(roi_geometry_df.head())

# Montage of all sessions/DMDs with ROI labels.
nrows = len(sessions_df)
fig, axes = plt.subplots(nrows, 2, figsize=(14, 4.2 * nrows), squeeze=False)

for r, (_, srow) in enumerate(sessions_df.iterrows()):
    sid = str(srow["session_id"])
    label = srow["session_label"]

    for c, dmd in enumerate([1, 2]):
        ax = axes[r, c]
        if sid in roi_image_cache and dmd in roi_image_cache[sid]:
            ref = roi_image_cache[sid][dmd]["ref"]
            masks = roi_image_cache[sid][dmd]["masks"]
            finite = ref[np.isfinite(ref)]
            vmin = np.nanpercentile(finite, 1)
            vmax = np.nanpercentile(finite, 99.5)
            ax.imshow(ref, cmap="gray", vmin=vmin, vmax=vmax)
            overlay_roi_masks(ax, masks)

            depth = srow.get(f"dmd{dmd}_depth_um", np.nan)
            depth_text = f" — {depth:.0f} µm" if np.isfinite(depth) else ""
            ax.set_title(f"{label} · DMD{dmd}{depth_text}")
        else:
            ax.set_title(f"{label} · DMD{dmd} unavailable")
        ax.set_axis_off()

fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIG_DIR / "all_sessions_roi_mask_montage.png", bbox_inches="tight")
plt.show()


## 11. Image/mask-based longitudinal registration

The recommended workflow remains semi-automatic:

1. Choose a reference session.
2. Estimate within-DMD translation between each session and the reference image.
3. Shift each session's masks into reference-session pixel coordinates.
4. Compute mask IoU and centroid distance against reference ROIs.
5. Use Hungarian assignment for suggestions.
6. Review/edit the output CSV and load manual overrides.

The exported map includes both the immutable `session_id` and the human-readable image-set/day label. This makes the same registration table reusable in later notebooks without relying on ROI number alone.


In [ ]:
def estimate_translation_to_reference(moving, reference, upsample_factor=10):
    # Return dy, dx shift that should be applied to moving to align it to reference.
    moving = np.asarray(moving, float)
    reference = np.asarray(reference, float)
    # Crop to common size.
    h = min(moving.shape[0], reference.shape[0])
    w = min(moving.shape[1], reference.shape[1])
    mov = moving[:h, :w]
    ref = reference[:h, :w]
    # High-pass-ish normalization to reduce illumination differences.
    mov = mov - ndimage.gaussian_filter(mov, 20)
    ref = ref - ndimage.gaussian_filter(ref, 20)
    try:
        from skimage.registration import phase_cross_correlation
        shift, error, phase = phase_cross_correlation(ref, mov, upsample_factor=upsample_factor, normalization=None)
        return float(shift[0]), float(shift[1]), float(error)
    except Exception:
        # Fallback: FFT phase correlation via scipy/numpy at integer precision.
        corr = signal.fftconvolve(ref, mov[::-1, ::-1], mode="same")
        y, x = np.unravel_index(np.nanargmax(corr), corr.shape)
        dy = y - corr.shape[0] // 2
        dx = x - corr.shape[1] // 2
        return float(dy), float(dx), np.nan


def shift_mask(mask, dy, dx):
    shifted = ndimage.shift(mask.astype(float), shift=(dy, dx), order=0, mode="constant", cval=0.0)
    return shifted > 0.5


def mask_iou(a, b):
    a = np.asarray(a, bool)
    b = np.asarray(b, bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union if union else np.nan


def centroid_distance(a, b):
    ya, xa = np.nonzero(a)
    yb, xb = np.nonzero(b)
    if len(xa) == 0 or len(xb) == 0:
        return np.nan
    return float(np.hypot(np.mean(xa) - np.mean(xb), np.mean(ya) - np.mean(yb)))


def suggest_registration_for_dmd(dmd, ref_session_id, iou_min=0.05, dist_max_px=30.0):
    ref_pack = roi_image_cache[ref_session_id][dmd]
    ref_img = ref_pack["ref"]
    ref_masks = ref_pack["masks"]
    rows = []
    shift_rows = []
    for sid in sessions_df["session_id"]:
        if sid not in roi_image_cache or dmd not in roi_image_cache[sid]:
            continue
        pack = roi_image_cache[sid][dmd]
        if sid == ref_session_id:
            dy, dx, err = 0.0, 0.0, 0.0
        else:
            dy, dx, err = estimate_translation_to_reference(pack["ref"], ref_img)
        shift_rows.append({"session_id": sid, "dmd": dmd, "dy_px": dy, "dx_px": dx, "registration_error": err})
        mov_masks = np.asarray([shift_mask(m, dy, dx) for m in pack["masks"]])
        cost = np.full((len(mov_masks), len(ref_masks)), 1e6, dtype=float)
        details = {}
        for i, m in enumerate(mov_masks):
            for j, r in enumerate(ref_masks):
                iou = mask_iou(m, r)
                dist = centroid_distance(m, r)
                valid = np.isfinite(iou) and np.isfinite(dist) and (iou >= iou_min or dist <= dist_max_px)
                # Prefer overlap, but allow centroid-based matching when masks are redrawn slightly differently.
                c = (1 - np.nan_to_num(iou, nan=0.0)) + 0.03 * np.nan_to_num(dist, nan=999)
                cost[i, j] = c if valid else 1e6
                details[(i, j)] = (iou, dist)
        if cost.size:
            ii, jj = optimize.linear_sum_assignment(cost)
            matched_ref = set()
            for i, j in zip(ii, jj):
                iou, dist = details[(i, j)]
                accept = cost[i, j] < 1e6
                cell_id = f"DMD{dmd}_cell{j:02d}" if accept else "UNMATCHED"
                matched_ref.add(j)
                rows.append({
                    "session_id": sid, "dmd": dmd, "roi": int(i),
                    "suggested_cell_id": cell_id,
                    "reference_roi": int(j) if accept else np.nan,
                    "iou_to_reference": iou,
                    "centroid_distance_px": dist,
                    "dy_px": dy, "dx_px": dx,
                    "auto_accept": bool(accept),
                    "manual_cell_id": "",
                    "include": True,
                    "notes": "",
                })
        # Add any unmatched moving ROIs explicitly.
        assigned = {(r["session_id"], r["dmd"], r["roi"]) for r in rows}
        for i in range(len(mov_masks)):
            if (sid, dmd, i) not in assigned:
                rows.append({
                    "session_id": sid, "dmd": dmd, "roi": int(i),
                    "suggested_cell_id": "UNMATCHED", "reference_roi": np.nan,
                    "iou_to_reference": np.nan, "centroid_distance_px": np.nan,
                    "dy_px": dy, "dx_px": dx, "auto_accept": False,
                    "manual_cell_id": "", "include": True, "notes": "",
                })
    return pd.DataFrame(rows), pd.DataFrame(shift_rows)

reg_tables = []
shift_tables = []

available_dmds = sorted({
    dmd
    for session_pack in roi_image_cache.values()
    for dmd in session_pack.keys()
})

for dmd in available_dmds:
    if (
        REFERENCE_SESSION_ID in roi_image_cache
        and dmd in roi_image_cache[REFERENCE_SESSION_ID]
    ):
        reg, shifts = suggest_registration_for_dmd(
            dmd,
            REFERENCE_SESSION_ID,
        )
        reg_tables.append(reg)
        shift_tables.append(shifts)

registration_suggestions_df = (
    pd.concat(reg_tables, ignore_index=True)
    if reg_tables else pd.DataFrame()
)
registration_shifts_df = (
    pd.concat(shift_tables, ignore_index=True)
    if shift_tables else pd.DataFrame()
)

registration_context = sessions_df[
    SESSION_CONTEXT_COLUMNS
].drop_duplicates("session_id")

if not registration_suggestions_df.empty:
    registration_suggestions_df = registration_suggestions_df.merge(
        registration_context,
        on="session_id",
        how="left",
    )
    registration_suggestions_df["depth_um"] = np.where(
        registration_suggestions_df["dmd"].eq(1),
        registration_suggestions_df["dmd1_depth_um"],
        registration_suggestions_df["dmd2_depth_um"],
    )

if not registration_shifts_df.empty:
    registration_shifts_df = registration_shifts_df.merge(
        registration_context,
        on="session_id",
        how="left",
    )

display(registration_shifts_df)
if not registration_suggestions_df.empty:
    display(
        registration_suggestions_df
        .sort_values(["dmd", "suggested_cell_id", "session_order", "roi"])
        .head(30)
    )

if SAVE_TABLES:
    registration_suggestions_df.to_csv(
        TABLE_DIR / "roi_identity_registration_suggestions.csv",
        index=False,
    )
    registration_shifts_df.to_csv(
        TABLE_DIR / "roi_image_registration_shifts.csv",
        index=False,
    )


## 12. Manual registration override table and reusable identity manifest

Edit `roi_identity_manual_overrides.csv` with one row per session/DMD/ROI. The stable key is:

`session_id,dmd,roi`

Useful context columns such as `session_label`, image set/day, and depth are included automatically. The final outputs are:

- `roi_identity_map_final.csv`: long-form canonical lookup for later notebooks;
- `roi_identity_map_wide.csv`: one row per registered neuron, one ROI-index column per image-set day.

Use `cell_id` as the cross-session neuron identity. Never use `session_label` alone as a database key.


In [ ]:
MANUAL_OVERRIDE_CSV = TABLE_DIR / "roi_identity_manual_overrides.csv"

if registration_suggestions_df.empty:
    raise RuntimeError("No registration suggestions were produced.")

template_columns = [
    "subject_id",
    "session_id",
    "session_label",
    "session_order",
    "image_set",
    "image_set_day_index",
    "dmd",
    "depth_um",
    "roi",
    "suggested_cell_id",
    "reference_roi",
    "iou_to_reference",
    "centroid_distance_px",
]

if not MANUAL_OVERRIDE_CSV.exists():
    template = registration_suggestions_df[
        [c for c in template_columns if c in registration_suggestions_df.columns]
    ].copy()
    template["manual_cell_id"] = ""
    template["include"] = True
    template["notes"] = ""
    template.to_csv(MANUAL_OVERRIDE_CSV, index=False)
    print("Wrote manual override template:", MANUAL_OVERRIDE_CSV)

manual = pd.read_csv(MANUAL_OVERRIDE_CSV)
key = ["session_id", "dmd", "roi"]
manual = manual.drop_duplicates(key)

manual_fields = key + ["manual_cell_id", "include", "notes"]
manual_fields = [c for c in manual_fields if c in manual.columns]

final_registration_df = (
    registration_suggestions_df
    .drop(columns=["manual_cell_id", "include", "notes"], errors="ignore")
    .merge(manual[manual_fields], on=key, how="left")
)

if "manual_cell_id" not in final_registration_df:
    final_registration_df["manual_cell_id"] = ""
if "include" not in final_registration_df:
    final_registration_df["include"] = True
if "notes" not in final_registration_df:
    final_registration_df["notes"] = ""

manual_id = (
    final_registration_df["manual_cell_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)
final_registration_df["cell_id"] = np.where(
    manual_id.ne(""),
    manual_id,
    final_registration_df["suggested_cell_id"],
)

# Robustly parse booleans edited in CSV software.
include_text = final_registration_df["include"].fillna(True).astype(str).str.lower()
final_registration_df["include"] = include_text.isin(["true", "1", "yes", "y"])
final_registration_df.loc[
    final_registration_df["cell_id"].eq("UNMATCHED"),
    "include",
] = False

final_registration_df = final_registration_df.sort_values(
    ["dmd", "cell_id", "session_order", "roi"]
).reset_index(drop=True)

# Convenient cross-day lookup: ROI index at each image-set day.
wide_source = final_registration_df[final_registration_df["include"]].copy()
roi_identity_wide_df = (
    wide_source
    .pivot_table(
        index=["subject_id", "dmd", "cell_id"],
        columns="session_label",
        values="roi",
        aggfunc="first",
    )
    .reset_index()
)
roi_identity_wide_df.columns.name = None

if SAVE_TABLES:
    final_registration_df.to_csv(
        TABLE_DIR / "roi_identity_map_final.csv",
        index=False,
    )
    roi_identity_wide_df.to_csv(
        TABLE_DIR / "roi_identity_map_wide.csv",
        index=False,
    )

display(final_registration_df.head(50))
display(roi_identity_wide_df)


## 13. Longitudinal metric table after registration

This merges the canonical ROI identity map back onto ROI-level electrophysiology metrics. The resulting table can be grouped by registered neuron, image set, day index, depth, or session.


In [ ]:
registration_merge_columns = [
    "session_id", "dmd", "roi", "cell_id", "include", "notes",
]

registered_metrics_df = metrics_df.merge(
    final_registration_df[registration_merge_columns],
    on=["session_id", "dmd", "roi"],
    how="left",
)
registered_metrics_df = registered_metrics_df[
    registered_metrics_df["include"].fillna(False)
].copy()

if SAVE_TABLES:
    registered_metrics_df.to_csv(
        TABLE_DIR / "asap8_registered_roi_ephys_metrics.csv",
        index=False,
    )

display(registered_metrics_df.head())
print("n registered cells:", registered_metrics_df["cell_id"].nunique())


## 14. Session-to-session stability plots

The x-axis uses the registry-derived labels A₀, A₁, B₀, and so on, ordered by the actual session chronology.


In [ ]:
def plot_longitudinal_metric(df, metric, min_sessions=2):
    sub = df.dropna(subset=[metric, "cell_id"]).copy()
    counts = sub.groupby("cell_id")["session_id"].nunique()
    keep = counts[counts >= min_sessions].index
    sub = sub[sub["cell_id"].isin(keep)]

    if sub.empty:
        print(f"No cells with >= {min_sessions} sessions for {metric}")
        return None, None

    session_axis = (
        df[["session_order", "session_label"]]
        .drop_duplicates()
        .sort_values("session_order")
    )
    order_to_label = dict(zip(
        session_axis["session_order"],
        session_axis["session_label"],
    ))

    fig, ax = plt.subplots(figsize=(9, 4.8))
    for cell_id, group in sub.groupby("cell_id"):
        group = group.sort_values("session_order")
        ax.plot(
            group["session_order"],
            group[metric],
            "-o",
            lw=1.2,
            ms=4,
            alpha=0.8,
            label=cell_id,
        )

    xticks = list(order_to_label)
    ax.set_xticks(xticks)
    ax.set_xticklabels(
        [order_to_label[x] for x in xticks],
        rotation=0,
        ha="center",
    )
    ax.set_xlabel("Image set and within-set day")
    ax.set_ylabel(metric)
    ax.set_title(f"Longitudinal {metric} by registered neuron")
    ax.legend(
        frameon=False,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8,
    )
    fig.tight_layout()
    return fig, ax


for metric in [
    "spike_rate_hz",
    "median_width_ms",
    "median_plateau_index",
    "plateau_burden_fraction",
]:
    fig, ax = plot_longitudinal_metric(registered_metrics_df, metric)
    if fig is not None and SAVE_FIGURES:
        fig.savefig(FIG_DIR / f"longitudinal_{metric}.png", bbox_inches="tight")
    plt.show()


## 15. Morphology annotation table from superstack/session context

This remains intentionally lightweight. The physiology notebook should not become a reconstruction pipeline, but it should carry enough morphology and depth context to interpret electrophysiological differences.

Suggested annotations:

- `layer_or_depth_label`: measured depth and/or cortical layer;
- `morphology_class`: bipolar, multipolar, bitufted, ambiguous;
- `soma_visible`, `dendrites_visible`, `axon_visible`: quick QC booleans;
- `superstack_or_figure_path`: link to the relevant morphology image;
- `notes`: branchiness, nearby labeled cells, mask quality, appearing/disappearing cells.


In [ ]:
MORPHOLOGY_CSV = TABLE_DIR / "registered_cell_morphology_annotations.csv"

if not MORPHOLOGY_CSV.exists():
    cells = (
        final_registration_df
        .loc[
            final_registration_df["include"].fillna(False),
            ["subject_id", "cell_id", "dmd", "depth_um"],
        ]
        .drop_duplicates(["subject_id", "cell_id", "dmd"])
        .sort_values(["dmd", "cell_id"])
    )
    morph = cells.copy()
    morph["layer_or_depth_label"] = ""
    morph["morphology_class"] = ""
    morph["soma_visible"] = ""
    morph["dendrites_visible"] = ""
    morph["axon_visible"] = ""
    morph["superstack_or_figure_path"] = ""
    morph["notes"] = ""
    morph.to_csv(MORPHOLOGY_CSV, index=False)
    print("Wrote morphology annotation template:", MORPHOLOGY_CSV)
else:
    morph = pd.read_csv(MORPHOLOGY_CSV)

display(morph)


## 16. Slide-ready summaries

These tables summarize electrophysiological phenotype by image-set day, recording depth, and registered neuron.


In [ ]:
slide_metrics = [
    "spike_rate_hz",
    "median_width_ms",
    "median_rise10_90_ms",
    "median_decay50_ms",
    "median_plateau_index",
    "compound_event_fraction_50ms",
    "plateau_burden_fraction",
    "snr_median_peak",
]

session_summary = (
    registered_metrics_df
    .groupby(
        [
            "session_label",
            "session_id",
            "image_set",
            "image_set_day_index",
            "dmd",
            "depth_um",
        ],
        dropna=False,
    )[slide_metrics]
    .agg(["median", "count"])
)
display(session_summary)

cell_summary = (
    registered_metrics_df
    .groupby(
        ["cell_id", "dmd", "depth_um"],
        dropna=False,
    )[slide_metrics]
    .agg(["median", "std", "count"])
)
display(cell_summary)

if SAVE_TABLES:
    session_summary.to_csv(TABLE_DIR / "slide_summary_by_image_set_day_dmd.csv")
    cell_summary.to_csv(TABLE_DIR / "slide_summary_by_registered_cell.csv")


## 17. Suggested lab-meeting figures from this notebook

For the opening electrophysiology section, the strongest narrative is:

1. **Registry/session overview**: A₀, A₁, B₀, etc., with recording depths and the number of tracked somata.
2. **Reference/mask montage**: the same cells across image-set days, with a small number appearing or disappearing.
3. **Detection overlay + spike-triggered waveforms**: ASAP8 resolves interpretable fast somatic events.
4. **Phenotype scatter**: event rate versus plateau index, with spike width and cortical depth as complementary encodings.
5. **Metric distributions by depth**: spike rate, width, plateau index, compound-event fraction, and plateau burden.
6. **Longitudinal tracks**: stable versus changing electrophysiological phenotype in registered neurons.
7. **Morphology callout**: one or two cells tying soma depth and broad morphology to spike phenotype.

### Canonical outputs for future notebooks

- `selected_session_registry_snapshot.csv`
- `roi_identity_map_final.csv`
- `roi_identity_map_wide.csv`
- `asap8_registered_roi_ephys_metrics.csv`
- `registered_cell_morphology_annotations.csv`

Load `roi_identity_map_final.csv` in later analyses and merge on `session_id`, `dmd`, and `roi`.
